# 🎓 Student Monitoring System Menu

**Module:** 25COP504 – Programming Fundamentals  
**Coursework Component:** Student Monitoring System  
**Student ID:** F517819  


- This application provides an interactive interface for preprocessing assessment data, retrieving student test results, analysing question-level performance, and identifying underperforming students using an SQLite database.   

  
- Relevant in-line comments and Markdown explanations are provided throughout the notebook to describe the purpose, structure, and functionality of each section of code.

### Imports

This section imports the libraries and modules needed for the student monitoring system.

- `ipywidgets` and `IPython.display` are used to build an interactive menu interface in the notebook 
- The four coursework modules are imported so the menu can call their functionality:
  - `CWPreprocessingManager` (Section 1.2)
  - `TestResultsManager` (Section 1.3)
  - `StudentPerformanceManager` (Section 1.4)
  - `UnderperformingStudentManager` (Section 1.5)

In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Import classes using exact filenames
try:
    from CWPreprocessing import CWPreprocessingManager
    from testResults import TestResultsManager
    from studentPerformance import StudentPerformanceManager
    from underPerforming import UnderperformingStudentManager
    
except ImportError as e:
    print(f"❌ ImportError: {e}")
    print("Check if the module exists and contains the correct class.")

### Database Setup (Managers)

This section initialises the system using a single SQLite database path.
`db_path = "CWDatabase.db"` defines the database path

In [2]:
# Initialise Database Managers
db_path = "CWDatabase.db"
preprocessor = CWPreprocessingManager(db_path)
test_manager = TestResultsManager(db_path)
perf_manager = StudentPerformanceManager(db_path)
underperf_manager = UnderperformingStudentManager(db_path)

### Preprocessing and storing data (Section 1.2)

This section implements the requirements of Section 1.2 using the
`CWPreprocessingManager` class and it's associated methods.

The following methods are used:
- `load_csv()` loads each raw CSV file into a separate DataFrame.
- `clean_columns()` renames columns to remove spaces, slashes, and question mark details.
- `fill_missing_values()` replaces null values with 0.
- `retain_highest_score()` keeps only the highest grade for students with multiple attempts.
- `drop_unnecessary_columns()` removes redundant columns such as *State* and *Time taken*.
- `convert_types()` ensures consistent data types for database storage.
- `adjust_grade()` standardises grades to a 0–100 scale.
- `save_tables()` creates SQLite tables and stores the cleaned DataFrames in `CWDatabase.db`.

Created dedicated output area for the application:
Error handling (`FileNotFoundError`) prevents the UI from crashing if any CSV is missing.

In [3]:
output = widgets.Output()

with output:
    csv_paths = [
        "TestResults/Formative_Test_1.csv",
        "TestResults/Formative_Test_2.csv",
        "TestResults/Formative_Test_3.csv",
        "TestResults/Formative_Test_4.csv",
        "TestResults/Formative_Mock_Test.csv",
        "TestResults/SumTest.csv"
    ]

    # Function to preprocess data (with error handling so the UI doesn't crash)
    dfs_original = []
    missing_files = []
    for path in csv_paths:
        try:
            dfs_original.append(preprocessor.load_csv(path))
        except FileNotFoundError:
            missing_files.append(path)

    if missing_files:
        for path in missing_files:
            print(f"❌ File not found: {path}")
        print("\n Database was NOT updated because some files are missing.")
    else:
        dfs_cleaned = [preprocessor.clean_columns(df) for df in dfs_original]
        dfs_cleaned = [preprocessor.fill_missing_values(df) for df in dfs_cleaned]
        dfs_cleaned = [preprocessor.retain_highest_score(df) for df in dfs_cleaned]
        dfs_cleaned = [preprocessor.drop_unnecessary_columns(df) for df in dfs_cleaned]
        dfs_cleaned = [preprocessor.convert_types(df) for df in dfs_cleaned]

        # Overall Grade scaling
        grade_max_scores = [6, 7, 6, 10, 100, 100]
        table_names = ["Test1", "Test2", "Test3", "Test4", "MockTest", "SumTest"]

        dfs_scaled = [
            preprocessor.adjust_grade(df, max_score)
            for df, max_score in zip(dfs_cleaned, grade_max_scores)
        ]

        tables_to_save = dict(zip(table_names, dfs_scaled))
        preprocessor.save_tables(tables_to_save)

        print("✅ Database is up-to-date.")

### Question Maximum Scores constants 

- Defines the maximum mark for each question in every test.
- This enables question-level percentage calculations for student performance analysis later (See **View Student Performance (Section 1.4)**)

In [4]:
# Question max scores (constants for studentPerformance)

QUESTION_MAX_SCORES = {
    "Test1": {"Q1": 1, "Q2": 1, "Q3": 1, "Q4": 1, "Q5": 1, "Q6": 1},
    "Test2": {"Q1": 1, "Q2": 1, "Q3": 1, "Q4": 2, "Q5": 1, "Q6": 1},
    "Test3": {"Q1": 1, "Q2": 1, "Q3": 1, "Q4": 1, "Q5": 1, "Q6": 1},
    "Test4": {"Q1": 5, "Q2": 5},
    "MockTest": {
        "Q1": 5, "Q2": 3, "Q3": 6, "Q4": 7, "Q5": 5,
        "Q6": 4, "Q7": 10, "Q8": 20, "Q9": 20, "Q10": 20
    },
    "SumTest": {
        "Q1": 5, "Q2": 3, "Q3": 6, "Q4": 7, "Q5": 4,
        "Q6": 5, "Q7": 15, "Q8": 15, "Q9": 15, "Q10": 10,
        "Q11": 4, "Q12": 5, "Q13": 6
    }
}

## User Interface

- The user interface provides an interactive, menu-driven layout that allows users to perform different analysis tasks within the system. It includes labelled input fields, dropdown menus, and an action button to execute the selected task.

- Only the inputs relevant to the selected task are displayed, while unnecessary fields are hidden. This reduces visual clutter, guides the user through the workflow, and minimises the risk of invalid input.

In [5]:
# Widgets
title = widgets.HTML("<h3>🎓 Student Monitoring System</h3>")

# Task Selector 
task_selector = widgets.ToggleButtons(
    options=[
        ("View Test Results", "test_results"),
        ("View Student Performance", "student_performance"),
        ("View Underperforming Students", "underperforming"),
    ],
    value="test_results",
    description="",
    style={"description_width": "60px"},
)
    
task_selector.layout = widgets.Layout(
    width="100%",
    margin="0 0 12px 0")

# Resercher ID input 
student_id_input = widgets.BoundedIntText(
    description="Researcher ID:",
    value=1,
    min=1,
    max=156,
    step=1,
    style={"description_width": "130px"},
    layout=widgets.Layout(width="260px")
)

# Test selection dropdown
test_name_dropdown = widgets.Dropdown(
    options=["Test1", "Test2", "Test3", "Test4", "MockTest", "SumTest"],
    description="Select Test:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="260px")
)

input_box = widgets.HBox(
    [student_id_input, test_name_dropdown],
    layout=widgets.Layout(gap="12px")
)

# Action button
run_button = widgets.Button(
    description="Run",
    icon="play",
    button_style="primary",
    layout=widgets.Layout(width="200px")
)


# Results Panel
output_box = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #ddd",
        padding="12px",
        margin="10px 0",
        width="100%"
    )
)

def set_visibility_for_task(task_value: str):
    """
    Shows/hides input widgets based on the chosen task.
    """
    if task_value == "underperforming":
        student_id_input.layout.display = "none"
        test_name_dropdown.layout.display = "none"
    elif task_value == "test_results":
        student_id_input.layout.display = ""
        test_name_dropdown.layout.display = "none"
    elif task_value == "student_performance":
        student_id_input.layout.display = ""
        test_name_dropdown.layout.display = ""

        
def set_button_style_for_task(task_value: str):
    """
    Updates action button label and sets style.
    """
    run_button.button_style = "info" 

    if task_value == "test_results":
        run_button.description = "View Test Results"
    elif task_value == "student_performance":
        run_button.description = "View Student Performance"
    elif task_value == "underperforming":
        run_button.description = "View Underperforming Students"


def set_status(message: str):
    """Update the status message shown to the user."""
    status.value = f"<span style='font-size:14px; font-weight:500;'>{message}</span>"


status = widgets.HTML("")

# Task change handler
def on_task_change(change):
    if change["name"] != "value":
        return
    task_value = change["new"]
    set_visibility_for_task(task_value)
    set_button_style_for_task(task_value)

    if task_value == "test_results":
        set_status("Choose a Researcher ID, then run to view all test grades")
    elif task_value == "student_performance":
        set_status("Choose a Researcher ID and a Test, then run to view question-level performance")
    else:
        set_status(
    "Identify underperforming students<br>"
    "<div style='margin-top:6px; margin-bottom:2px;'>"
    "Underperforming criteria:"
    "</div>"
    "<ul style='margin-top:2px; margin-bottom:0; padding-left:18px;'>"
    "<li>Summative grade &lt; 50%</li>"
    "<li>Minimum formative attempts = 3</li>"
    "<li>At least 2 formative grades &lt; 50%</li>"
    "</ul>"
)
task_selector.observe(on_task_change)

### Button Action: Run Selected Task (Sections 1.3–1.5)

The system uses a single action button whose behaviour depends on the task selected in the `Task Selector`.  

#### View Test Results (Section 1.3)

When `View Test Results` is selected, the system retrieves and displays all assessment grades for a given student.

Methods used from `TestResultsManager`:
- `fetch_grade_from_table()` – retrieves grades from the database  
- `display_grades()` – displays grades in a table  
- `plot_grades()` – visualises grades using Matplotlib  

---

#### View Student Performance (Section 1.4)

When `View Student Performance` is selected, the system performs a question-level performance analysis for a specific test.

Methods used from `StudentPerformanceManager`:
- `load_test_table()` – loads the selected test from the database  
- `standardise_scores()` – converts question scores to percentages  
- `student_performance()` – computes performance metrics  
- `display_performance()` – presents results in tabular form  
- `plot_performance()` – visualises performance results  

---

####  View Underperforming Students (Section 1.5)

When `View Underperforming Students` is selected, the system identifies students who meet predefined underperformance criteria.

Methods used from `UnderperformingStudentManager`:
- `merge_formative_summative()` – combines assessment data  
- `identify_underperforming_students()` – applies underperformance criteria  
- `print_underperforming_students()` – displays results in text form  
- `plot_underperforming_students()` – visualises underperforming students  

In [6]:
def run_selected_task(button):
    """
    Runs the selected analysis task and displays the results in the output area.
    
    Args: 
        button (ipywidgets.button): button widget that triggered the callback
    """
    with output_box:
        clear_output(wait=True)

        task_value = task_selector.value

        if task_value == "test_results":
            researcher_id = student_id_input.value
            tables = ["Test1", "Test2", "Test3", "Test4", "MockTest", "SumTest"]
            grades = {t: test_manager.fetch_grade_from_table(researcher_id, t) for t in tables}
            test_manager.display_grades(grades, researcher_id)
            test_manager.plot_grades(grades, researcher_id)

        elif task_value == "student_performance":
            researcher_id = student_id_input.value
            test_name = test_name_dropdown.value

            df = perf_manager.load_test_table(test_name)
            df_std = perf_manager.standardise_scores(df, test_name, QUESTION_MAX_SCORES)
            perf_df = perf_manager.student_performance(df_std, researcher_id)

            perf_manager.display_performance(perf_df, test_name, researcher_id)
            perf_manager.plot_performance(perf_df, test_name, researcher_id)

        elif task_value == "underperforming":
            formative_tables = ["Test1", "Test2", "Test3", "Test4", "MockTest"]
            summative_table = "SumTest"

            merged_df = underperf_manager.merge_formative_summative(formative_tables, summative_table)
            under_df = underperf_manager.identify_underperforming_students(merged_df)

            underperf_manager.print_underperforming_students(under_df)
            underperf_manager.plot_underperforming_students(under_df)

run_button.on_click(run_selected_task)

### Layout

- The layout is constructed using `widgets.VBox` and `widgets.HBox` toorganise controls and the shared `output` area.
- Results are rendered using the `widgets.Output` container to maintain a clean interface.
- The layout is initialised by setting the correct input visibility, button label, and status message based on the default selected task before rendering the interface.

In [7]:
# Layout (with spacing)
controls = widgets.VBox(
    [
        title,
        task_selector,
        input_box,
        widgets.HBox([run_button], layout=widgets.Layout(margin="10px 0 0 0")),
        status
    ],
    layout=widgets.Layout(gap="10px")
)

# Initialise correct visibility + button style + status
set_visibility_for_task(task_selector.value)
set_button_style_for_task(task_selector.value)
set_status("Choose a Researcher ID, then run to view all test grades.")

display(widgets.VBox([controls, output_box], layout=widgets.Layout(gap="10px")))